<a href="https://colab.research.google.com/github/JuanZapa7a/Medical-Image-Processing/blob/main/PIM_Challenge/PIM_Challenge_Student_Practice_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# UPCT Medical Image Segmentation Challenge 2026-27
## Practice 6

**Course:** Medical Image Processing (521104007)

**Professor:** Juan Zapata

> **How this notebook series works:** a new notebook is released every week (Practice 6 to 10) with the content of that session. **This first notebook (P6) is the base of your project**: complete it, and from next week on copy the new cells of each practice and paste them **at the end of this same notebook** — do not start a new one every week or paste practices you have already done. By the time you reach P10 you will have your own complete notebook with the 5 practices: that is the one you will submit in the virtual classroom.
>
> Each practice opens in a **new Colab runtime**, so every day you will have to run all the cells of your notebook from the beginning (including those from previous weeks) — you do not need to redo anything: the checkpoints on Drive detect that they already exist and load directly instead of retraining.

## Session Guide (2 hours per session)
| Practice | Dates (Group A / B) | Session Objective | Visual Checkpoint |
|----------|----------------------|-------------------|-------------------|
| ▶ **P6** | 28 Oct - 2 Nov | EDA, Dataset and RLE format | 6 images with masks + RLE OK |
| **P7** | 9-11 Nov | U-Net baseline and 1st submission | Loss plots + Kaggle submission |
| **P8** | 16-18 Nov | Data augmentation and improvement | Baseline vs Augmented comparison |
| **P9** | 23-25 Nov | Inference, threshold and errors | 5 normal images + 2 error cases |
| **P10** | 30 Nov-2 Dec | TTA, final submission and defense | Best Dice Score + oral defense |

> **Golden Rule:** According to Art. 7.5 of the UPCT Assessment Regulations, attendance and validation of the Checkpoint in the classroom is mandatory to pass the practice.



## Download the Competition Dataset

The dataset for this challenge is **private** and is only available inside the competition on Kaggle.

### Instructions:
1. **Accept the invitation** to the challenge that you received by email (or access it directly):
   [UPCT Medical Image Segmentation Challenge](https://www.kaggle.com/competitions/upct-medical-image-segmentation-challenge-2026-27)

2. Inside the competition, go to the **"Data"** tab (in the top menu).

3. Click on **"Download All"** or download the file **`upct-medical-image-segmentation-challenge-2026-27.zip`** (approx. 150 MB).

4. **Upload the ZIP file** to your Google Drive in a folder called **`PIM_Challenge`**:
   - Go to [Google Drive](https://drive.google.com)
   - Create the `PIM_Challenge` folder (if it does not exist)
   - Upload the ZIP there

> **IMPORTANT:**
> - **DO NOT change the name of the ZIP file** (it must keep its original name)
> - **DO NOT unzip the ZIP** before uploading it (we will do it from Colab)
> - If you cannot access the competition, contact the professor: juan.zapata@upct.es

## Dataset structure

Once unzipped, the dataset will have this structure:



In [ ]:
# ============================================================
# VERIFICATION AND EXPLORATION OF THE DOWNLOADED DATASET
# ============================================================

# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# 2. Install tree to visualize folders
!apt-get install -y tree > /dev/null 2>&1

# 3. Locate the ZIP in PIM_Challenge
import os
import zipfile
import shutil
from pathlib import Path

DRIVE_FOLDER = '/content/drive/MyDrive/PIM_Challenge'
EXTRACT_DIR = Path('/content/PIM_Challenge_dataset')

print(f"\n Searching for dataset in: {DRIVE_FOLDER}\n")

# List the contents of the folder
if os.path.exists(DRIVE_FOLDER):
    print(" Contents of PIM_Challenge:")
    for f in os.listdir(DRIVE_FOLDER):
        size = os.path.getsize(os.path.join(DRIVE_FOLDER, f)) / (1024*1024)
        print(f"   - {f} ({size:.1f} MB)")
else:
    print(f" The folder {DRIVE_FOLDER} does not exist")
    raise FileNotFoundError(DRIVE_FOLDER)

# Look for the ZIP
zip_files = [f for f in os.listdir(DRIVE_FOLDER) if f.endswith('.zip')]

if not zip_files:
    print("\n There is no ZIP file in PIM_Challenge")
    print(" Upload the ZIP you downloaded from Kaggle to that folder")
    raise FileNotFoundError("No ZIP in PIM_Challenge")

ZIP_PATH = os.path.join(DRIVE_FOLDER, zip_files[0])
print(f"\n ZIP found: {zip_files[0]}")

# 4. Unzip
if not EXTRACT_DIR.exists():
    print(f"\n Unzipping {zip_files[0]}...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall('/content')
    print("Unzipping completed.\n")
else:
    print(f"\nDataset already unzipped in {EXTRACT_DIR}\n")

# 5. Reorganize if necessary (Kaggle sometimes unzips directly in /content/)
print("Verifying the dataset structure...")

if not (EXTRACT_DIR / 'train').exists() and Path('/content/train').exists():
    print(" Dataset unzipped in /content/. Reorganizing...")

    if not EXTRACT_DIR.exists():
        EXTRACT_DIR.mkdir()

    items_to_move = ['train', 'test', 'sample_submission.csv']
    for item in items_to_move:
        src = Path(f'/content/{item}')
        dst = EXTRACT_DIR / item

        if src.exists():
            if dst.exists():
                if dst.is_dir():
                    shutil.rmtree(dst)
                else:
                    dst.unlink()

            shutil.move(str(src), str(dst))
            print(f"    Moved: {item}")
        else:
            print(f"    Not found: {item}")

    print(" Reorganization completed.\n")
elif (EXTRACT_DIR / 'train').exists():
    print(" Correct structure detected.\n")
else:
    print(" Expected structure not found. Looking for train/test folders...\n")

# 6. Show the dataset structure
print("="*60)
print(" DATASET STRUCTURE")
print("="*60)
!tree -L 3 /content/PIM_Challenge_dataset --dirsfirst

# 7. Detailed statistics
print("\n" + "="*60)
print(" DETAILED STATISTICS")
print("="*60)

import glob

train_imgs = glob.glob(f'{EXTRACT_DIR}/train/images/*.png')
train_masks = glob.glob(f'{EXTRACT_DIR}/train/masks/*.png')
test_imgs = glob.glob(f'{EXTRACT_DIR}/test/images/*.png')

# Count by class
train_benign = len([f for f in train_imgs if 'benign' in f])
train_malignant = len([f for f in train_imgs if 'malignant' in f])
train_normal = len([f for f in train_imgs if 'normal' in f])

test_benign = len([f for f in test_imgs if 'benign' in f])
test_malignant = len([f for f in test_imgs if 'malignant' in f])
test_normal = len([f for f in test_imgs if 'normal' in f])

print(f"\n TRAIN SET: {len(train_imgs)} images")
print(f"   ├─ Benign:     {train_benign:3d} images")
print(f"   ├─ Malignant:  {train_malignant:3d} images")
print(f"   └─ Normal:     {train_normal:3d} images")
print(f"    Masks:   {len(train_masks)} files")

print(f"\n TEST SET: {len(test_imgs)} images (WITHOUT masks)")
print(f"   ├─ Benign:     {test_benign:3d} images")
print(f"   ├─ Malignant:  {test_malignant:3d} images")
print(f"   └─ Normal:     {test_normal:3d} images")

has_csv = os.path.exists(f'{EXTRACT_DIR}/sample_submission.csv')
print(f"\nsample_submission.csv: {' Exists' if has_csv else ' Does not exist'}")

# 8. Integrity check
print("\n" + "="*60)
print(" INTEGRITY CHECK")
print("="*60)

checks = [
    (len(train_imgs) == 546, f"Train images: {len(train_imgs)}/546"),
    (len(train_masks) == 546, f"Train masks: {len(train_masks)}/546"),
    (len(test_imgs) == 234, f"Test images: {len(test_imgs)}/234"),
    (has_csv, "sample_submission.csv exists"),
]

all_ok = True
for ok, desc in checks:
    print(f"   {'OK' if ok else 'noOK'} {desc}")
    if not ok:
        all_ok = False

# Final check
if all_ok:
    print("\n Dataset correct and ready to work with!")
else:
    print("\n There are problems in the dataset. Review the files.")

# PRACTICE 6: Dataset Exploration and RLE Format
## Single session (28 Oct - Wednesday - and 2 Nov - Monday -)

### Session objectives:
1. Understand the structure of the BUSI dataset (benign, malignant, normal)
2. Visualize images and segmentation masks
3. Understand the RLE format (Run-Length Encoding) for Kaggle
4. Create the DataLoader for training

### Clinical context
The BUSI dataset contains 780 breast ultrasound images:
- **Benign (437)**: Non-cancerous lesions
- **Malignant (210)**: Cancerous lesions
- **Normal (133)**: Healthy tissue (no lesion)

**Clinical challenge**: The model must be able to say "there is no tumor" in normal images.

> **CHECKPOINT P6:** Show the professor the visualization of the images with their overlaid masks and the verification that your `mask_to_rle`/`rle_to_mask` functions reconstruct the original mask correctly.

### Estimated time: 2 hours

## Block 6.1: Semantic Segmentation in Medical Images
### Classification vs Detection vs Segmentation

| Task | Question it answers | Output |
|------|---------------------|--------|
| **Classification** | Is there a tumor? | Label: "benign" / "malignant" |
| **Detection** | Where is the tumor? | Bounding box |
| **Segmentation** | Which pixels are tumor? | Pixel-by-pixel mask |

### Why segmentation and not classification?
- In medicine, the **size and shape** of the tumor matter (staging)
- It allows computing **volume**, **irregular borders**, **invasion**
- It is the basis of **radiomics** and computer-assisted diagnosis

### The specific challenge: Breast ultrasound (BUS)
- **Noisy** and **low-contrast** images
- **Small** tumors (sometimes < 5% of the image)
- Artifacts: acoustic shadows, reverberations
- **"normal" class**: the model must learn NOT to detect anything

## Task 6.1: Loading and structuring the Dataset
Before training any model, we need to organize the information in memory.
We will use the **Pandas** library to create a DataFrame that acts as an index of our dataset, relating each image to its class and its mask.

### Instructions:
1. Define the base path of the dataset: `/content/PIM_Challenge_dataset`.
2. Walk through the `train/images` folder looking for all `.png` images.
3. For each image, extract its **class** (`benign`, `malignant` or `normal`) based on the directory name.
4. Look for the path of its corresponding mask in the `train/masks` folder. *(Hint: the mask name is usually the same as the image but adding `_mask` at the end).*
5. Store all this information in a **Pandas DataFrame** with the columns: `class`, `image_path`, `mask_path` and `filename`.
6. Display on screen the total number of images and the class distribution (how many of each type: benign, malignant and normal).

> **Recommended libraries:** `pandas`, `pathlib`, `os`.



In [ ]:
# ============================================================
# TASK 6.1: LOADING THE DATASET IN PANDAS
# ============================================================

DATA_DIR = Path('/content/PIM_Challenge_dataset')

# WRITE YOUR CODE HERE
# 1. Create an empty list to store the data
# 2. Iterate over the classes: ['benign', 'malignant', 'normal']
# 3. For each class, look for the images in DATA_DIR / 'train' / 'images'
# 4. Build the path of the corresponding mask
# 5. Add a dictionary with the information to the list
# 6. Convert the list into a DataFrame

data = []

# Your code here...


df_train = pd.DataFrame(data)

print(f"Dataset loaded: {len(df_train)} training images.")
print("\n Class distribution in Train:")
print(df_train['class'].value_counts())

## Block 6.2: The BUSI Dataset (Breast Ultrasound Images)
### Origin and composition
- Published by **Al-Dhabyani et al. (2020)**
- 780 breast ultrasound images
- 3 classes:
  - **Benign (437)**: fibroadenomas, cysts...
  - **Malignant (210)**: carcinomas
  - **Normal (133)**: healthy tissue

### Class imbalance

>  **Question to think about:** Why are there more benign than malignant images in clinical reality?

### The "Hidden Challenge": normal images
Most segmentation competitions assume that **all images contain the object to segment**. In this challenge:
- **normal** images have an **empty** mask (all black)
- If your model paints a tumor where there is none → **FALSE POSITIVE**
- Clinical consequence: unnecessary biopsy, anxiety, healthcare cost

### Metrics: why don't we use Accuracy?
Imagine a model that says "there is no tumor" on ALL images:
- Accuracy: ~85% (because 85% of the pixels are background)
- Dice Score: **0** (it has not detected any tumor)

That is why in medical segmentation we use **Dice Score** or **IoU**.

> **Question to think about:** "If you had to diagnose a patient, what would worry you more: a false positive or a false negative? Why?"



## Task 6.2: Visualization and the "Hidden Clinical Challenge"
We cannot trust the data blindly. Let's visualize the data to make sure the masks are aligned with the images and understand the clinical difficulty of the challenge.

### Instructions:
1. Select **1 random image from each class** (benign, malignant, normal) from your DataFrame.
2. Create a **3 rows x 3 columns** figure using `matplotlib` (one row per class).
3. For each image, load and display in the 3 columns:
   * **Column 1 (Original):** The image in RGB. *(Remember that `cv2.imread` reads it in BGR, so you will have to convert it).*
   * **Column 2 (Real Mask):** The mask in grayscale.
   * **Column 3 (Overlay):** Copy the original image and **paint in RED `[255, 0, 0]`** the pixels where the mask indicates there is a tumor.
4. Analyze the images of the **"normal"** class. Write a `print()` at the end explaining:
   * What do you see in their masks?
   * What would happen clinically if your AI model predicts a tumor (red pixels) in one of these images?

> **CHECKPOINT P6.1:** When you have the 3x3 figure generated, call the professor to validate your visualization and your clinical conclusion.

> **Recommended libraries:** `cv2`, `matplotlib.pyplot`, `numpy`.

In [ ]:
# ============================================================
# TASK 6.2: VISUALIZATION OF EXAMPLES
# ============================================================

# WRITE YOUR CODE HERE

# 1. Select 1 random image from each class
# YOUR CODE HERE

# 2. Create the figure
# YOUR CODE HERE

# 3. Iterate over the samples and display the 3 columns
# YOUR CODE HERE

# plt.tight_layout()
# plt.show()

# 4. Analysis of normal images
# YOUR CODE HERE

# Check if the masks of the normal images are empty
# YOUR CODE HERE

# print(f"'normal' images with a completely black (empty) mask: {empty_masks}/{len(normal_samples)}")
# print("\n CLINICAL CONCLUSION:")
# print("If your model predicts a tumor in a 'normal' image, it will be committing a FALSE POSITIVE.")
# print("In medicine, this can lead to unnecessary biopsies and anxiety in the patient.")
# print("Your model must learn NOT to segment anything when there is no lesion.")

## Block 6.3: Submission Formats and Metrics
### Why RLE and not PNG?
Kaggle needs a **text** format to:
1. **Compression**: a 512x512 mask = 262,144 pixels → in RLE it can be just 20 numbers
2. **Automatic validation**: easy to parse and compare
3. **Multiple objects**: several tumors can be encoded in a single image

### Key metrics in Kaggle

**Dice Score (F1 Score):**
$$Dice = \frac{2 \cdot |A \cap B|}{|A| + |B|}$$

**IoU (Intersection over Union):**
$$IoU = \frac{|A \cap B|}{|A \cup B|}$$

**Relationship:** $Dice = \frac{2 \cdot IoU}{1 + IoU}$

| Dice Score | Clinical interpretation |
|------------|-------------------------|
| 0.9 - 1.0 | Excellent segmentation |
| 0.7 - 0.9 | Good (typical in competitions) |
| 0.5 - 0.7 | Acceptable, can be improved |
| < 0.5 | Model not clinically useful |

### Calculate manually:
"Calculate by hand the Dice of this example:"

    Prediction: 100 tumor pixels
    Ground truth: 100 tumor pixels
    Intersection: 80 pixels
    Answer: Dice = 2·80/(100+100) = 0.80

## Task 6.3: The RLE Format (Run-Length Encoding)
Kaggle does not accept masks as PNG images for submissions. Instead, it requires masks to be sent in text format using **Run-Length Encoding (RLE)**.

### What is RLE?
It is a very simple compression method: instead of storing each pixel, we store **sequences of equal pixels**.

**Example:**
Imagine a 5x5 mask (25 pixels) where the center pixels are tumor (1) and the rest is background (0):

0 0 0 0 0

0 1 1 1 0

0 1 1 1 0

0 1 1 1 0

0 0 0 0 0


In RLE format, this is expressed as:
`"7 3 12 3 17 3"`

**What does it mean?**
- `7 3` → Starting at pixel 7, there are 3 tumor pixels
- `12 3` → Starting at pixel 12, there are 3 tumor pixels
- `17 3` → Starting at pixel 17, there are 3 tumor pixels

> **IMPORTANT:** Kaggle reads the images **by columns** (top to bottom, then the next column), not by rows. And the indices start at **1**, not 0.

### Instructions:
1. Implement the function `mask_to_rle(mask)` that:
   - Receives a binary mask (numpy array of 0s and 1s)
   - Flattens it by columns (use `.T.flatten()`)
   - Detects the changes from 0 to 1 and from 1 to 0
   - Returns a string with the RLE format (pairs of numbers separated by spaces)

2. Implement the function `rle_to_mask(rle_string, height, width)` that:
   - Receives an RLE string and the dimensions of the image
   - Reconstructs the original binary mask
   - Returns a numpy array of shape `(height, width)`

3. **Test your functions:**
   - Create a test mask (you can use a real mask from the dataset or create an artificial one)
   - Convert it to RLE with `mask_to_rle()`
   - Reconstruct it with `rle_to_mask()`
   - Verify that the original and the reconstructed masks are **identical** (use `np.array_equal()`)

4. **Visualize the result:**
   - Display 3 images: original mask, reconstructed mask, and the difference (it should be all black if it works correctly)

> **Hint:** To detect the changes in the pixel sequence, you can use `np.where(pixels[1:] != pixels[:-1])`.

> **CHECKPOINT P6.2:** Show the professor that your RLE functions work correctly (the difference between the original and the reconstructed mask must be zero).

In [ ]:
# ============================================================
# TASK 6.3: RLE IMPLEMENTATION
# ============================================================

import numpy as np

# WRITE YOUR CODE HERE

def mask_to_rle(mask):
    """
    Converts a binary mask to RLE format.

    Args:
        mask: binary numpy array (0s and 1s) of shape (height, width)

    Returns:
        str: string with the RLE format (pairs of numbers separated by spaces)
    """
    # Your code here
    pass


def rle_to_mask(rle_string, height, width):
    """
    Converts an RLE string to a binary mask.

    Args:
        rle_string: string with RLE format
        height: height of the image
        width: width of the image

    Returns:
        binary numpy array of shape (height, width)
    """
    # Your code here
    pass


# ============================================================
# TESTING THE FUNCTIONS (do not modify)
# ============================================================

# Load a real mask from the dataset
sample_mask_path = df_train[df_train['class'] == 'benign'].iloc[0]['mask_path']
original_mask = cv2.imread(sample_mask_path, cv2.IMREAD_GRAYSCALE)
original_mask_bin = (original_mask > 127).astype(np.uint8)

print(f"Original mask: shape {original_mask_bin.shape}, tumor pixels: {original_mask_bin.sum()}")

# Convert to RLE
rle_encoded = mask_to_rle(original_mask_bin)
print(f"RLE encoded (first 50 characters): {rle_encoded[:50]}...")

# Reconstruct from RLE
reconstructed_mask = rle_to_mask(rle_encoded, original_mask_bin.shape[0], original_mask_bin.shape[1])
print(f"Reconstructed mask: shape {reconstructed_mask.shape}, tumor pixels: {reconstructed_mask.sum()}")

# Verify that they are identical
are_equal = np.array_equal(original_mask_bin, reconstructed_mask)
print(f"\nAre they identical? {are_equal}")

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(original_mask_bin, cmap='gray')
axes[0].set_title("Original Mask")
axes[0].axis('off')

axes[1].imshow(reconstructed_mask, cmap='gray')
axes[1].set_title("Reconstructed from RLE")
axes[1].axis('off')

difference = np.abs(original_mask_bin.astype(int) - reconstructed_mask.astype(int))
axes[2].imshow(difference, cmap='hot')
axes[2].set_title(f"Difference (should be black)\nSum of differences: {difference.sum()}")
axes[2].axis('off')

plt.tight_layout()
plt.show()

if are_equal:
    print("\nIt works perfectly! Your RLE functions are correct.")
else:
    print("\nThere are differences. Review your implementation.")